### Libraries

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import pyogrio
import rasterio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
NAIP_DIR = Path("/Volumes/Storage/naip")  # external drive


def check(path, script):
    """True if the file exists; otherwise say which script creates it."""
    if path.exists():
        return True
    print(f"Not downloaded yet: {path}\n  -> run python scripts/download/{script}")
    return False


def sample_vector(path, layer=None, n=5):
    """Feature count, CRS, and the first n rows of a GeoPackage/GeoJSON layer (reads only n rows)."""
    info = pyogrio.read_info(path, layer=layer)
    print(f"{path.name}{f' [{layer}]' if layer else ''}: {info['features']:,} features, CRS {info['crs']}")
    return gpd.read_file(path, layer=layer, rows=n)


def sample_csv(path, n=5):
    with open(path) as f:
        n_rows = sum(1 for _ in f) - 1
    print(f"{path.name}: {n_rows:,} rows")
    return pd.read_csv(path, nrows=n)


def describe_raster(path):
    with rasterio.open(path) as src:
        print(f"{path.name}: {src.width} x {src.height} px, {src.count} bands {src.descriptions}, "
              f"{src.res[0]:.6g} {src.crs.linear_units if src.crs.is_projected else 'deg'} pixels, CRS {src.crs}")

# Data Acquisition

Sources and URLs: `data/data_sources.txt`

## 0. County boundaries

In [ ]:
COUNTIES = RAW / "core_counties.geojson"

if check(COUNTIES, "download_naip.py --list-years"):
    counties = gpd.read_file(COUNTIES)
    display(counties[["NAME", "GEOID", "ALAND", "geometry"]])

## 1. NAIP imagery

In [ ]:
NAIP_INDEX = NAIP_DIR / "naip_index.geojson"

if check(NAIP_INDEX, "download_naip.py --list-years"):
    naip_index = gpd.read_file(NAIP_INDEX)
    naip_index["downloaded"] = naip_index["path"].map(lambda p: (NAIP_DIR / p).exists())
    display(naip_index.groupby("year").agg(tiles=("id", "size"), downloaded=("downloaded", "sum"),
                                          gsd_m=("gsd", "first")))
    display(naip_index.head())

In [ ]:
# One downloaded tile per year
if NAIP_INDEX.exists():
    for year, group in naip_index[naip_index["downloaded"]].groupby("year"):
        describe_raster(NAIP_DIR / group["path"].iloc[0])

## 2. Building footprints

In [ ]:
OSM = RAW / "osm" / "osm_core.gpkg"

if check(OSM, "download_osm.py"):
    display(sample_vector(OSM, layer="buildings"))
    display(sample_vector(OSM, layer="industrial"))

In [ ]:
MS_BUILDINGS = RAW / "ms_buildings" / "ms_buildings_core.gpkg"

if check(MS_BUILDINGS, "download_ms_buildings.py"):
    display(sample_vector(MS_BUILDINGS, layer="buildings"))

## 3. Parcels and land use

In [ ]:
PARCELS = RAW / "parcels" / "parcels_core.gpkg"

if check(PARCELS, "download_parcels.py"):
    display(sample_vector(PARCELS, layer="parcels"))

## 4. Truck traffic (HPMS)

In [ ]:
TRAFFIC = RAW / "traffic" / "hpms_core.gpkg"

if check(TRAFFIC, "download_traffic.py"):
    layers = sorted(pyogrio.list_layers(TRAFFIC)[:, 0])
    print("Layers:", layers)
    display(sample_vector(TRAFFIC, layer=layers[-1]))

## 5. Satellite NO2 (TROPOMI)

In [ ]:
NO2_FILES = sorted((RAW / "no2").glob("tropomi_no2_*.tif"))

if NO2_FILES:
    for path in NO2_FILES:
        describe_raster(path)
else:
    check(RAW / "no2" / "tropomi_no2_<year>.tif", "download_no2_tropomi.py")

## 6. Ground air quality

In [ ]:
AQS_NO2 = RAW / "air_monitors" / "aqs_daily_no2.csv"
AQS_PM25 = RAW / "air_monitors" / "aqs_daily_pm25.csv"

for path in [AQS_NO2, AQS_PM25]:
    if check(path, "download_air_monitors.py"):
        display(sample_csv(path))

In [ ]:
FAQSD_ANNUAL = RAW / "faqsd" / "faqsd_pm25_annual.csv"
FAQSD_DAILY = sorted((RAW / "faqsd").glob("faqsd_pm25_daily_*.csv"))

if check(FAQSD_ANNUAL, "download_faqsd.py"):
    display(sample_csv(FAQSD_ANNUAL))
if FAQSD_DAILY:
    display(sample_csv(FAQSD_DAILY[-1]))

In [ ]:
PM25_SAT_FILES = sorted((RAW / "pm25_satellite").glob("acag_pm25_*.tif"))

if PM25_SAT_FILES:
    for path in PM25_SAT_FILES:
        describe_raster(path)
else:
    check(RAW / "pm25_satellite" / "acag_pm25_<year>.tif", "download_pm25_satellite.py")

## 7. Asthma ER visits (manual, NJSHAD)

In [ ]:
HEALTH_FILES = sorted((RAW / "health").glob("*.csv"))

if HEALTH_FILES:
    for path in HEALTH_FILES:
        display(sample_csv(path))
else:
    print("No files in data/raw/health/ yet; export from https://www-doh.nj.gov/doh-shad/")

## 8. Demographics (Census ACS)

In [ ]:
TRACTS = RAW / "census" / "tracts_acs_2024.gpkg"

if check(TRACTS, "download_census.py"):
    display(sample_vector(TRACTS, layer="tracts"))

## 9. Environmental justice layers

In [ ]:
OBC = RAW / "ej" / "overburdened_communities.gpkg"
EJSCREEN = RAW / "ej" / "ejscreen_2024_tracts_core.csv"

if check(OBC, "download_ej_layers.py"):
    display(sample_vector(OBC, layer="obc"))
if check(EJSCREEN, "download_ej_layers.py"):
    display(sample_csv(EJSCREEN))

## 10. Planned warehouses (manual, optional)

In [ ]:
PLANNED = RAW / "planned_warehouses.csv"

if PLANNED.exists():
    display(sample_csv(PLANNED))
else:
    print("No planned_warehouses.csv yet (optional)")